In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
from skimage.io import imread
from skimage.filters import threshold_triangle, threshold_yen
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy.ndimage import binary_dilation
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIGURATION — edit these
# ─────────────────────────────────────────────
INPUT_DIR  = r'E:\ND2\tiff'
OUTPUT_DIR = r'E:\ND2\tiff\output'
MIN_CELL   = 500
MAX_CELL   = 6000

# ─────────────────────────────────────────────
# SEGMENTATION FUNCTION
# ─────────────────────────────────────────────
def segment_file(filepath, output_dir, min_cell=500, max_cell=6000):
    img = imread(str(filepath))
    img4 = img[:, :, 3]   # membrane channel
    
    # ── Membrane segmentation ──────────────────
    img_smooth     = ndi.gaussian_filter(img4, sigma=2)
    thresh         = threshold_triangle(img_smooth)
    mem            = img_smooth > thresh
    mem_holefilled = ~ndi.binary_fill_holes(~(~mem))

    i   = 10
    SE  = ((np.mgrid[:i,:i][0] - np.floor(i/2))**2 +
           (np.mgrid[:i,:i][1] - np.floor(i/2))**2) <= np.floor(i/2)**2
    pad = i + 1
    mem_padded  = np.pad(~mem_holefilled, pad, mode='reflect')
    mem_dilated = binary_dilation(mem_padded, structure=SE)
    mem_filled  = ndi.binary_fill_holes(mem_dilated)
    mem_final   = ~mem_filled[pad:-pad, pad:-pad]

    # ── Distance transform & seeds ────────────
    cell_labels      = ndi.label(~mem_final)[0]
    dist_trans       = ndi.distance_transform_edt(~mem_final)
    dist_trans_smooth= ndi.gaussian_filter(dist_trans, sigma=2)

    seed_coords      = peak_local_max(dist_trans_smooth, min_distance=10)
    seeds            = np.zeros_like(dist_trans_smooth, dtype=bool)
    seeds[tuple(seed_coords.T)] = True
    seeds_labeled    = ndi.label(seeds)[0]
    seeds_labeled_dil= ndi.maximum_filter(seeds_labeled, size=10)

    # ── Watershed ─────────────────────────────
    ws = watershed(-dist_trans_smooth, seeds_labeled, mask=~mem_final)

    # ── Remove border cells ───────────────────
    border_mask = ndi.binary_dilation(np.zeros(ws.shape, dtype=bool), border_value=1)
    clean_ws    = np.where(
        np.array([np.any(np.logical_and(ws == cid, border_mask))
                  for cid in range(ws.max()+1)])[ws],
        0, ws)

    # ── Size filter & relabel ─────────────────
    valid_ids = [cid for cid in np.unique(clean_ws)[1:]
                 if min_cell < np.sum(clean_ws == cid) < max_cell]
    clean_ws2 = np.zeros_like(clean_ws)
    for new_id, old_id in enumerate(valid_ids, start=1):
        clean_ws2[clean_ws == old_id] = new_id
    new_count = len(valid_ids)

    # ── Cell edges ────────────────────────────
    edges = np.zeros_like(clean_ws2)
    for cid in np.unique(clean_ws2)[1:]:
        mask  = clean_ws2 == cid
        edges[np.logical_xor(mask, ndi.binary_erosion(mask, iterations=5))] = cid

    # ── Summary figure ────────────────────────
    label_color = '#99ccff'
    def add_panel(ax, image, title_num=None, cmap='gray', vmin=None, vmax=None,
                  overlay=None, overlay_cmap='prism', extra_text=None):
        ax.set_facecolor('black')
        ax.imshow(image, interpolation='none', cmap=cmap, vmin=vmin, vmax=vmax)
        if overlay is not None:
            ax.imshow(np.ma.array(overlay, mask=overlay==0),
                      interpolation='none', cmap=overlay_cmap)
        if title_num is not None:
            ax.text(0.02, 0.98, f'{title_num}', transform=ax.transAxes,
                    fontsize=16, color=label_color, fontweight='bold', va='top', ha='left')
        if extra_text is not None:
            ax.text(0.98, 0.98, extra_text, transform=ax.transAxes,
                    fontsize=12, color='white', va='top', ha='right')
        ax.axis('off')

    fig = plt.figure(figsize=(18, 18))
    fig.patch.set_facecolor('black')
    gs  = fig.add_gridspec(3, 3, hspace=0.05, wspace=0.05)

    positions = {
        'center': (1,1), 1:(0,0), 2:(0,1), 3:(0,2),
        4:(1,2), 5:(2,2), 6:(2,1), 7:(2,0), 8:(1,0)
    }

    ax_c = fig.add_subplot(gs[positions['center']])
    ax_c.imshow(img4, interpolation='none', cmap='gray')
    ax_c.text(0.5, 0.98, 'RAW', transform=ax_c.transAxes,
              fontsize=14, color='white', va='top', ha='center', fontweight='bold')
    ax_c.axis('off')

    panels = {
        1: (img_smooth,           dict(cmap='gray')),
        2: (mem,                  dict(cmap='gray')),
        3: (mem_holefilled,       dict(cmap='gray')),
        4: (cell_labels,          dict(cmap='inferno')),
        5: (dist_trans_smooth,    dict(cmap='viridis')),
        6: (img_smooth,           dict(cmap='gray', overlay=seeds_labeled_dil, overlay_cmap='prism')),
        7: (np.zeros_like(clean_ws2),           dict(cmap='gray', overlay=ws, overlay_cmap='prism')),
        8: (np.zeros_like(edges), dict(cmap='gray', overlay=edges,
                                       extra_text=f'Cells: {new_count}')),
    }

    for num, (image, kwargs) in panels.items():
        ax = fig.add_subplot(gs[positions[num]])
        add_panel(ax, image, title_num=num, **kwargs)

    stem = filepath.stem
    plt.suptitle(stem, color='white', fontsize=18, fontweight='bold', y=0.995)
    outpath = Path(output_dir) / f'{stem}_segmentation.png'
    plt.savefig(outpath, dpi=150, bbox_inches='tight', facecolor='black')
    plt.close()
    print(f"  ✓ {stem}: {new_count} cells → {outpath.name}")
    return new_count


# ─────────────────────────────────────────────
# BATCH LOOP
# ─────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
tif_files = sorted(Path(INPUT_DIR).glob('*.tif'))
print(f"Found {len(tif_files)} files in {INPUT_DIR}\n")

summary = {}
for fp in tif_files:
    print(f"Processing: {fp.name}")
    summary[fp.name] = segment_file(fp, OUTPUT_DIR, MIN_CELL, MAX_CELL)

print("\n── Batch complete ──")
for fname, count in summary.items():
    print(f"  {fname}: {count} cells")

# ─────────────────────────────────────────────
# AVERAGE CELL COUNTS PER ND2 SOURCE FILE
# Groups locations (e.g. sample_loc01.tif, sample_loc02.tif)
# back to their parent ND2 file and reports the average cell count.
# ─────────────────────────────────────────────
import re
from collections import defaultdict

nd2_groups = defaultdict(list)
for fname, count in summary.items():
    # Strip trailing _locNN suffix to recover the ND2 base name
    base = re.sub(r'_loc\d+\.tif$', '', fname)
    nd2_groups[base].append(count)

print("\n── Average cell count per ND2 file ──")
for base, counts in sorted(nd2_groups.items()):
    avg  = sum(counts) / len(counts)
    locs = len(counts)
    print(f"  {base}: {avg:.1f} cells/location  "
          f"(locations: {locs},  individual counts: {counts})")    

Found 1056 files in E:\ND2\tiff

Processing: C66_SC_101_1_10X_loc01.tif
  ✓ C66_SC_101_1_10X_loc01: 152 cells → C66_SC_101_1_10X_loc01_segmentation.png
Processing: C66_SC_101_2_10X_loc01.tif
  ✓ C66_SC_101_2_10X_loc01: 460 cells → C66_SC_101_2_10X_loc01_segmentation.png
Processing: C66_SC_101_2_40X_loc01.tif
  ✓ C66_SC_101_2_40X_loc01: 212 cells → C66_SC_101_2_40X_loc01_segmentation.png
Processing: C66_SC_101_2_40X_loc02.tif
  ✓ C66_SC_101_2_40X_loc02: 346 cells → C66_SC_101_2_40X_loc02_segmentation.png
Processing: C66_SC_101_2_40X_loc03.tif
  ✓ C66_SC_101_2_40X_loc03: 358 cells → C66_SC_101_2_40X_loc03_segmentation.png
Processing: C66_SC_101_2_40X_loc04.tif
  ✓ C66_SC_101_2_40X_loc04: 367 cells → C66_SC_101_2_40X_loc04_segmentation.png
Processing: C66_SC_101_2_40X_loc05.tif
  ✓ C66_SC_101_2_40X_loc05: 389 cells → C66_SC_101_2_40X_loc05_segmentation.png
Processing: C66_SC_101_2_40X_loc06.tif
  ✓ C66_SC_101_2_40X_loc06: 283 cells → C66_SC_101_2_40X_loc06_segmentation.png
Processing: C66